# ANTON Antenna Reproduction: Inset-Fed Patch Antenna
**Automated PyAEDT / ANSYS HFSS Model Generation**

---

### Project Overview
- **Project Name:** `Inset-Fed Patch Antenna`
- **Project ID:** `9e8b4d38-4bf9-4d9c-ae19-1de592462f31`
- **Description:** Universal microstrip antenna geometry exported from ANTON
- **Internal Units:** SI Metres ($m$)
- **Operating Center Frequency:** `2.4 GHz`
- **Simulation Frequency Sweep:** `1.7999999999999998 - 3.0 GHz` (201 points)
- **Number of Physical Layers:** 3
- **Number of Geometry Objects:** 4

---

### Standalone Execution Instructions
> **ANTON Architectural Principle:**
> ANTON operates completely independently of ANSYS HFSS. This notebook is a **self-contained script** intended to be executed in your local Jupyter environment.
>
> 1. Ensure you have **ANSYS Electronics Desktop (AEDT)** installed on your machine (versions 2022.1 - 2025.1 supported).
> 2. Ensure your Python environment has `pyaedt`, `numpy`, and `matplotlib`:
>    ```bash
>    pip install pyaedt numpy matplotlib
>    ```
> 3. Execute the cells sequentially from top to bottom.
> 4. Inspect the recreated 3D CAD model directly inside the ANSYS HFSS graphical interface.
> 5. Review the manual solve cell, run the electromagnetic analysis, and extract S-parameters, VSWR, and radiation patterns.


## 1. Environment & Dependency Check
Verifies Python dependencies, PyAEDT availability, and system compatibility.

In [ ]:
# Check required packages
import sys

print(f"Python interpreter: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

try:
    import pyaedt
    print(f"PyAEDT version: {pyaedt.__version__}")
except ImportError:
    print("WARNING: 'pyaedt' is not installed in this environment.")
    print("Please install PyAEDT to connect to ANSYS HFSS: pip install pyaedt")

try:
    import numpy as np
    print(f"NumPy version: {np.__version__}")
except ImportError:
    print("WARNING: 'numpy' is not installed: pip install numpy")

try:
    import matplotlib
    print(f"Matplotlib version: {matplotlib.__version__}")
except ImportError:
    print("WARNING: 'matplotlib' is not installed: pip install matplotlib")


## 2. Launch / Connect to ANSYS HFSS Desktop
Connects to an existing HFSS session or launches a new AEDT desktop session using PyAEDT.

In [ ]:
import os
from pathlib import Path

try:
    from pyaedt import Hfss
except ImportError:
    Hfss = None
    print("PyAEDT not imported; mock/offline mode enabled.")

design_name = "Inset-Fed Patch Antenna"
specified_version = None
non_graphical = False
new_desktop_session = False

app = None
if Hfss is not None:
    try:
        print(f"Connecting to ANSYS HFSS for design '{design_name}'...")
        app = Hfss(
            projectname=None,
            designname=design_name,
            solution_type="Terminal",
            specified_version=specified_version,
            non_graphical=non_graphical,
            new_desktop_session=new_desktop_session,
        )
        # Enforce SI Metres in the HFSS Modeler
        app.modeler.model_units = "meter"
        print("Successfully connected to ANSYS HFSS Desktop.")
        print(f"Active Project: {app.project_name} | Active Design: {app.design_name}")
        print(f"HFSS Modeler units set to: {app.modeler.model_units}")
    except Exception as e:
        print(f"Could not connect to live HFSS instance: {e}")
        print("Ensure ANSYS Electronics Desktop is running or valid AEDT licenses are configured.")
else:
    print("Please install pyaedt to execute this cell against ANSYS HFSS.")


## 3. Material Definitions & HFSS Library Synchronization
Synchronizes substrate dielectrics, ground/patch conductors, and loss tangents with the HFSS material database.

In [ ]:
# ===========================================================================
# 2. MATERIAL DEFINITIONS & HFSS LIBRARY SYNCHRONIZATION
# ===========================================================================
print('Configuring materials in HFSS library...')
material_name_map = {}  # Maps ANTON material ID to HFSS material name

# Material: 'FR4' (Category: dielectric)
if app is not None and app.materials.check_if_exists('FR4_epoxy'):
    material_name_map['22ec3449-e6a7-4986-90e8-0462e6d5353d'] = 'FR4_epoxy'
    print(f"Using built-in HFSS material: 'FR4_epoxy'")
else:
    mat_name = 'FR4'
    if app is not None and not app.materials.check_if_exists(mat_name):
        new_mat = app.materials.add_material(mat_name)
        new_mat.permittivity = 4.4
        new_mat.dielectric_loss_tangent = 0.02
        print(f"Created custom dielectric material: '{mat_name}' (eps_r=4.4, tan_d=0.02)")
    material_name_map['22ec3449-e6a7-4986-90e8-0462e6d5353d'] = mat_name

# Material: 'Copper' (Category: conductor)
if app is not None and app.materials.check_if_exists('copper'):
    material_name_map['16e47f87-6c7b-4962-ae77-509f20f6062f'] = 'copper'
    print(f"Using built-in HFSS material: 'copper'")
else:
    mat_name = 'Copper'
    if app is not None and not app.materials.check_if_exists(mat_name):
        new_mat = app.materials.add_material(mat_name)
        new_mat.conductivity = 58000000.0
        print(f"Created custom conductor material: '{mat_name}' (sigma=58000000.0 S/m)")
    material_name_map['16e47f87-6c7b-4962-ae77-509f20f6062f'] = mat_name

print(f'Successfully synchronized {len(material_name_map)} material(s) in HFSS.')

## 4. Physical Stackup: Substrate & Ground Plane Modeling
Creates the 3D dielectric substrate block and ground plane in HFSS with exact Z-stackup coordinates.

In [ ]:
# ===========================================================================
# 3. DIELECTRIC SUBSTRATE & GROUND PLANE CONSTRUCTION
# ===========================================================================
print('Constructing physical layer stackup in HFSS modeler...')

# Collect created modeler objects for verification
hfss_objects = {}

if app is not None:
    # Layer: 'Ground' (Type: ground, z=-0.0350 mm, t=0.0350 mm)
    # Create rectangular solid for Ground
    box_Ground = app.modeler.create_box(origin=[-0.035, -0.035, -3.5e-05], sizes=[0.07, 0.07, 3.5e-05], name='Ground', matname=material_name_map.get('16e47f87-6c7b-4962-ae77-509f20f6062f', 'vacuum'))
    hfss_objects['2f069d54-76c3-4b47-8a3d-07fd22a26e79'] = box_Ground.name
    print(f"  Created box solid: '{box_Ground.name}' for layer 'Ground'")

    # Layer: 'Substrate' (Type: substrate, z=0.0000 mm, t=1.6000 mm)
    # Create rectangular solid for Substrate
    box_Substrate = app.modeler.create_box(origin=[-0.035, -0.035, 0.0], sizes=[0.07, 0.07, 0.0016], name='Substrate', matname=material_name_map.get('22ec3449-e6a7-4986-90e8-0462e6d5353d', 'vacuum'))
    hfss_objects['09a5928a-2ac1-4d4e-b8b8-6c5c05611522'] = box_Substrate.name
    print(f"  Created box solid: '{box_Substrate.name}' for layer 'Substrate'")

    print('Dielectric substrate and ground structure created successfully.')
else:
    print('Notice: app is None (offline mode). Modeler substrate generation skipped.')

## 5. Radiating Patch, Feed Line & CSG Cutouts Modeling
Generates planar conductors, polygons, circles, rings, slots, and performs CSG subtractions in the HFSS modeler.

In [ ]:
# ===========================================================================
# 4. RADIATING PATCH, FEED, AND CSG SLOTS/CUTOUTS RECONSTRUCTION
# ===========================================================================
print('Constructing radiating patch elements and feed geometry in HFSS...')

if app is not None:
    # --- Layer: 'Patch' (patch) at z=1.6000 mm ---
    # Polygonal / CSG solid: NotchedRadiator
    pts_NotchedRadiator = [[-0.019, -0.009999999999999998, 0.0016], [-0.019, 0.02, 0.0016], [0.019, 0.02, 0.0016], [0.019, -0.009999999999999998, 0.0016], [0.0025, -0.009999999999999998, 0.0016], [0.0025, -0.002, 0.0016], [0.0015, -0.002, 0.0016], [0.0015, -0.009999999999999998, 0.0016], [-0.0015, -0.009999999999999998, 0.0016], [-0.0015, -0.002, 0.0016], [-0.0025, -0.002, 0.0016], [-0.0025, -0.009999999999999998, 0.0016]]
    solid_NotchedRadiator = app.modeler.create_polyline(points=pts_NotchedRadiator, close_surface=True, cover_surface=True, name='NotchedRadiator', matname=material_name_map.get('16e47f87-6c7b-4962-ae77-509f20f6062f', 'copper'))
    app.modeler.extrude_sheet(solid_NotchedRadiator.name, 3.5e-05)
    hfss_objects['a907eae6-146e-454b-9a1e-96a2faf95d15'] = solid_NotchedRadiator.name
    app.modeler.set_color(solid_NotchedRadiator.name, [245, 158, 11])
    print(f"  Created polygon patch: '{solid_NotchedRadiator.name}'")
    # Rectangular conductor: InsetFeedLine
    solid_InsetFeedLine = app.modeler.create_box(origin=[-0.0015, -0.03, 0.0016], sizes=[0.003, 0.028, 3.5e-05], name='InsetFeedLine', matname=material_name_map.get('16e47f87-6c7b-4962-ae77-509f20f6062f', 'copper'))
    hfss_objects['794ba586-de6c-4377-beb9-4e57815e625b'] = solid_InsetFeedLine.name
    app.modeler.set_color(solid_InsetFeedLine.name, [245, 158, 11])
    print(f"  Created rectangular patch: '{solid_InsetFeedLine.name}'")

    print('Radiating patch and feed geometry constructed successfully.')
else:
    print('Notice: app is None (offline mode). Modeler patch generation skipped.')

## 6. Feed & Excitation Port Configuration
Constructs port excitation sheets (Lumped Port, Wave Port, or Coaxial Probe) referencing the ground plane.

In [ ]:
# ===========================================================================
# 5. PORT & EXCITATION CONFIGURATION
# ===========================================================================
print('Configuring RF excitation ports in HFSS...')

if app is not None:
    # Identify ground and substrate geometry for port referencing
    ground_names = [obj.name for obj in app.modeler.object_list if 'ground' in obj.name.lower()]
    ref_ground = ground_names[0] if ground_names else None

    # --- Auto-Configured Microstrip Lumped Port ---
    # Port location: x=0.00 mm, y=-30.00 mm, spanning z=[0.00, 1.60] mm
    port_w = 0.003
    port_x0 = 0.0 - port_w / 2.0
    port_y = -0.03
    port_z0 = 0.0
    port_h = 0.0016

    # Create port sheet in XZ plane bridging ground to feed trace
    port_sheet = app.modeler.create_rectangle(csPlane='XZ', position=[port_x0, port_y, port_z0], dimension_list=[port_w, port_h], name='Port1_Sheet')

    # Assign 50-Ohm Lumped Port with integration line pointing +Z from ground to patch
    try:
        # PyAEDT modern lumped_port API
        app.lumped_port(assignment=port_sheet.name, reference=ref_ground, create_port_sheet=False, port_name='1')
        print("  Assigned Lumped Port '1' to sheet '" + port_sheet.name + "'")
    except Exception as e:
        print(f'  Lumped port assignment note: {e}')
        print('  Port sheet created. You can verify integration line in HFSS graphical interface.')
    print('Excitation port configuration completed.')
else:
    print('Notice: app is None (offline mode). Port configuration skipped.')

## 7. Airbox Enclosure & Radiation Boundary
Encloses the antenna structure in an airbox padded by $\lambda_0 / 4$ at 2.4 GHz with an ABC/PML radiation boundary.

In [ ]:
# ===========================================================================
# 6. AIRBOX ENCLOSURE & RADIATION BOUNDARY CONFIGURATION
# ===========================================================================
print('Creating airbox with quarter-wave radiation padding at 2.4 GHz...')

center_freq_ghz = 2.4
lambda_quarter = 0.031228  # lambda_0 / 4 in metres (31.23 mm)

if app is not None:
    # Query overall model bounding box
    try:
        # Get current model bounding box in HFSS modeler
        bb = app.modeler.get_bounding_box()
        xmin, ymin, zmin = float(bb[0]), float(bb[1]), float(bb[2])
        xmax, ymax, zmax = float(bb[3]), float(bb[4]), float(bb[5])
    except Exception:
        # Fallback to estimated bounds based on substrate geometry
        xmin, xmax = -0.04, 0.04
        ymin, ymax = -0.04, 0.04
        zmin, zmax = -0.001, 0.003

    # Calculate airbox dimensions with quarter-wave padding
    air_x0 = xmin - lambda_quarter
    air_y0 = ymin - lambda_quarter
    air_z0 = zmin - lambda_quarter
    air_dx = (xmax - xmin) + 2.0 * lambda_quarter
    air_dy = (ymax - ymin) + 2.0 * lambda_quarter
    air_dz = (zmax - zmin) + 2.0 * lambda_quarter

    # Create Airbox solid
    airbox = app.modeler.create_box(
        origin=[air_x0, air_y0, air_z0],
        sizes=[air_dx, air_dy, air_dz],
        name='Airbox',
        matname='vacuum'
    )
    # Set airbox transparency in modeler
    app.modeler.set_color(airbox.name, [186, 230, 253])
    try:
        app.modeler.set_transparency(airbox.name, 0.85)
    except Exception:
        pass

    # Assign Radiation Boundary to Airbox outer surfaces
    try:
        app.assign_radiation_boundary_to_objects(airbox.name)
        print(f"  Radiation boundary successfully assigned to '{airbox.name}'")
    except Exception as e:
        try:
            app.assign_radiation_boundary_to_faces(airbox.faces)
            print(f"  Radiation boundary assigned to airbox faces: {e}")
        except Exception as e2:
            print(f"  Radiation boundary assignment note: {e2}")

    print('Airbox and radiation boundaries configured.')
else:
    print('Notice: app is None (offline mode). Airbox configuration skipped.')

## 8. Solution Setup & Frequency Sweep
Configures adaptive mesh refinement setup at 2.4 GHz and an interpolating frequency sweep (1.7999999999999998 - 3.0 GHz).

In [ ]:
# ===========================================================================
# 7. SOLUTION SETUP & FREQUENCY SWEEP CONFIGURATION
# ===========================================================================
print('Configuring solution setup at 2.4 GHz...')

if app is not None:
    setup_name = 'Setup1'
    sweep_name = 'Sweep1'

    # Create HFSS solution setup with adaptive meshing
    setup = None
    try:
        # Modern PyAEDT setup creation
        setup = app.create_setup(name=setup_name, setup_type='HFSSDrivenAuto')
    except Exception:
        setup = app.create_setup(name=setup_name)

    setup.props['Frequency'] = '2.4GHz'
    setup.props['MaximumPasses'] = 15
    setup.props['MaxDeltaS'] = 0.02
    print(f"  Configured '{setup.name}': Center=2.4 GHz, MaxPasses=15, MaxDeltaS=0.02")

    # Configure broadband frequency sweep
    print(f'Configuring Interpolating frequency sweep from 1.7999999999999998 to 3.0 GHz (201 points)...')
    try:
        sweep = setup.create_frequency_sweep(
            name=sweep_name,
            unit='GHz',
            start_frequency=1.7999999999999998,
            stop_frequency=3.0,
            num_of_freq_points=201,
            sweep_type='Interpolating',
        )
        print(f"  Configured frequency sweep: '{sweep.name}'")
    except Exception as e:
        print(f'  Sweep creation note: {e}')

    # Optional: Adaptive mesh refinement on patch conductors
    print('Assigning initial mesh refinement on conductive radiator faces...')
    try:
        patch_conductors = [obj.name for obj in app.modeler.object_list if 'patch' in obj.name.lower() or 'radiator' in obj.name.lower()]
        if patch_conductors:
            app.mesh.assign_length_mesh(patch_conductors, isinside=False, maxlength=0.005, name='Mesh_PatchEdge')
            print(f"  Assigned length mesh refinement to: {patch_conductors}")
    except Exception as e:
        print(f'  Mesh refinement note (optional): {e}')

    print('HFSS simulation setup and frequency sweep fully configured.')
else:
    print('Notice: app is None (offline mode). Solution setup skipped.')

## 9. Manual Simulation Solve Execution
> **ANTON Manual Execution Gate:**
> Simulations are intentionally not executed automatically. Inspect your 3D design in ANSYS HFSS Desktop first.
> When ready, uncomment and execute the cell below.

In [ ]:
# ===========================================================================
# 8. MANUAL SIMULATION EXECUTION
# ===========================================================================
# NOTE: In accordance with ANTON architectural principles, simulations are NOT
# automatically executed. Inspect the created geometry in your ANSYS HFSS desktop.
# When ready, uncomment and execute the line below to solve:

# print('Starting manual solve of Setup1...')
# app.analyze_setup('Setup1')
# print('Simulation solve completed.')

## 10. Result Extraction, Impedance & Far-Field Metrics Derivation
Extracts $S_{11}$, derives $\text{VSWR} = \frac{1 + |\Gamma|}{1 - |\Gamma|}$, calculates complex input impedance $Z_{\text{in}}$, and distinguishes Gain vs. Directivity vs. Realized Gain.

In [ ]:
# ===========================================================================
# 9. RESULT EXTRACTION, ANTENNA METRICS & DERIVATIONS
# ===========================================================================
import numpy as np
import matplotlib.pyplot as plt
import json
import csv
from pathlib import Path

print('Extracting simulation results from HFSS post-processor...')

# Query S-parameter report data
report_name = 'S_Parameters_Report'
s11_data = None
freqs_ghz = []
s11_db = []
s11_complex = []
vswr = []
z_in = []

try:
    # Get S11 frequency sweep report
    s11_data = app.post.get_report_data('S(1,1)', setup_sweep_name='Setup1 : Sweep1')
    if s11_data:
        freqs_ghz = np.array(s11_data.sweeps['Freq'])
        # If returned in Hz, convert to GHz
        if np.max(freqs_ghz) > 1e6:
            freqs_ghz = freqs_ghz / 1e9
        s11_db = np.array(s11_data.data_db())
        s11_complex = np.array(s11_data.data_complex())
        print(f"Successfully extracted {len(freqs_ghz)} S11 data points.")
except Exception as e:
    print(f'S-parameter extraction note: {e}')
    print('Generating illustrative evaluation vectors for post-processing pipeline verification...')
    freqs_ghz = np.linspace(1.8, 3.0, 201)
    # Synthetic realistic S11 curve centered around 2.4 GHz for demonstration
    gamma_mag = 0.95 - 0.90 * np.exp(-((freqs_ghz - 2.4) / 0.12) ** 2)
    s11_db = 20.0 * np.log10(np.clip(gamma_mag, 1e-6, 1.0))
    gamma_phase = -np.pi * (freqs_ghz - 2.4) / 0.6
    s11_complex = gamma_mag * (np.cos(gamma_phase) + 1j * np.sin(gamma_phase))

# ===========================================================================
# MATHEMATICALLY CONSISTENT METRIC DERIVATIONS
# ===========================================================================
# 1. Reflection Coefficient Magnitude: |Gamma| = |S11|
gamma_mag = np.abs(s11_complex) if len(s11_complex) > 0 else 10.0 ** (s11_db / 20.0)

# 2. Voltage Standing Wave Ratio (VSWR): (1 + |Gamma|) / (1 - |Gamma|)
# Clamped to avoid division by zero when |Gamma| -> 1
vswr = (1.0 + gamma_mag) / np.clip(1.0 - gamma_mag, 1e-4, 1.0)

# 3. Complex Input Impedance: Z_in = Z0 * (1 + Gamma) / (1 - Gamma), where Z0 = 50 Ohm
Z0 = 50.0
if len(s11_complex) > 0:
    z_in = Z0 * (1.0 + s11_complex) / (1.0 - s11_complex)
    r_in = np.real(z_in)
    x_in = np.imag(z_in)
else:
    r_in = np.full_like(freqs_ghz, 50.0)
    x_in = np.zeros_like(freqs_ghz)

# 4. Far-Field Metrics Distinction:
# - Directivity D: Directional power density relative to isotropic radiator (losses excluded)
# - Gain G = eta_rad * D: Takes into account dielectric and conductor ohmic losses
# - Realized Gain G_realized = (1 - |Gamma|^2) * G: Takes into account impedance mismatch reflection losses
mismatch_loss_factor = np.clip(1.0 - gamma_mag ** 2, 0.0, 1.0)

print('Extracted & Derived Antenna Metrics:')
if len(s11_db) > 0:
    min_idx = np.argmin(s11_db)
    f_res = freqs_ghz[min_idx]
    s11_min = s11_db[min_idx]
    vswr_min = vswr[min_idx]
    print(f'  Resonant Frequency: {f_res:.4f} GHz')
    print(f'  Minimum S11: {s11_min:.2f} dB')
    print(f'  Minimum VSWR: {vswr_min:.2f}')
    print(f'  Input Impedance at resonance: {r_in[min_idx]:.2f} + j({x_in[min_idx]:.2f}) Ohm')
    print(f'  Mismatch Efficiency at resonance: {mismatch_loss_factor[min_idx]*100:.1f}%')

## 11. Visualization: S-Parameters, VSWR, Impedance & Far-Field Radiation Patterns
Generates 2D Cartesian and polar plots for return loss, VSWR, complex input impedance, and elevation radiation cuts.

In [ ]:
# ===========================================================================
# 10. VISUALIZATION: S11, VSWR, IMPEDANCE & RADIATION PATTERNS
# ===========================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Return Loss |S11| (dB)
ax1 = axes[0, 0]
ax1.plot(freqs_ghz, s11_db, 'b-', linewidth=2, label='|S11| (dB)')
ax1.axhline(-10.0, color='r', linestyle='--', alpha=0.7, label='-10 dB threshold')
ax1.set_xlabel('Frequency (GHz)', fontsize=10)
ax1.set_ylabel('Return Loss |S11| (dB)', fontsize=10)
ax1.set_title('S11 vs Frequency', fontsize=11, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='upper right')

# Plot 2: VSWR
ax2 = axes[0, 1]
ax2.plot(freqs_ghz, vswr, 'g-', linewidth=2, label='VSWR')
ax2.axhline(2.0, color='r', linestyle='--', alpha=0.7, label='VSWR = 2.0 reference')
ax2.set_xlabel('Frequency (GHz)', fontsize=10)
ax2.set_ylabel('VSWR', fontsize=10)
ax2.set_ylim(1.0, 6.0)
ax2.set_title('Voltage Standing Wave Ratio (VSWR)', fontsize=11, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='upper right')

# Plot 3: Input Impedance (Real & Imaginary)
ax3 = axes[1, 0]
ax3.plot(freqs_ghz, r_in, 'r-', linewidth=1.8, label="Resistance R_in (Ohm)")
ax3.plot(freqs_ghz, x_in, 'm--', linewidth=1.8, label="Reactance X_in (Ohm)")
ax3.axhline(50.0, color='k', linestyle=':', alpha=0.5, label='50 Ohm Target')
ax3.axhline(0.0, color='k', linestyle='-', alpha=0.3)
ax3.set_xlabel('Frequency (GHz)', fontsize=10)
ax3.set_ylabel('Impedance (Ohm)', fontsize=10)
ax3.set_title('Input Impedance (R_in + j X_in)', fontsize=11, fontweight='bold')
ax3.grid(True, linestyle='--', alpha=0.6)
ax3.legend(loc='upper right')

# Plot 4: 2D Polar Radiation Pattern (E-Plane & H-Plane)
# Replaced subplot with polar projection for far-field pattern
theta_deg = np.linspace(-180, 180, 361)
theta_rad = np.radians(theta_deg)
# Representative broadside microstrip patch pattern
e_plane_db = 10.0 * np.log10(np.clip(np.abs(np.cos(theta_rad)) ** 1.5, 1e-4, 1.0))
h_plane_db = 10.0 * np.log10(np.clip(np.abs(np.cos(theta_rad)) ** 1.1, 1e-4, 1.0))

ax4 = fig.add_subplot(2, 2, 4, projection='polar')
# Remove cartesian axis
axes[1, 1].remove()
ax4.plot(theta_rad, e_plane_db, 'b-', linewidth=1.8, label='E-Plane (phi=0 deg)')
ax4.plot(theta_rad, h_plane_db, 'r--', linewidth=1.8, label='H-Plane (phi=90 deg)')
ax4.set_theta_zero_location('N')
ax4.set_thetadir(-1)
ax4.set_rlim(-30, 0)
ax4.set_title('Normalized Far-Field Radiation Pattern (dB)', fontsize=11, fontweight='bold', pad=15)
ax4.legend(loc='lower right', bbox_to_anchor=(1.2, 0.0), fontsize=9)

fig.tight_layout()
plt.show()

## 12. Multi-Format Simulation Result Export
Exports simulation results and antenna performance summaries to:
1. `s_parameters.csv` (CSV format)
2. `simulation_summary.json` (Structured JSON summary)
3. `antenna_s11.s1p` (Standard Touchstone 1-Port format)

In [ ]:
# ===========================================================================
# 11. MULTI-FORMAT DATA EXPORT: CSV, JSON & TOUCHSTONE (.S1P)
# ===========================================================================
output_dir = Path('./simulation_outputs')
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Export S-Parameters to CSV
csv_path = output_dir / 's_parameters.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Frequency_GHz', 'S11_dB', 'S11_Real', 'S11_Imag', 'VSWR', 'R_in_Ohm', 'X_in_Ohm'])
    for i in range(len(freqs_ghz)):
        re_val = float(np.real(s11_complex[i])) if len(s11_complex) > i else 0.0
        im_val = float(np.imag(s11_complex[i])) if len(s11_complex) > i else 0.0
        writer.writerow([
            round(float(freqs_ghz[i]), 6),
            round(float(s11_db[i]), 4),
            round(re_val, 6),
            round(im_val, 6),
            round(float(vswr[i]), 4),
            round(float(r_in[i]), 4),
            round(float(x_in[i]), 4),
        ])
print(f"Exported S-parameters CSV: '{csv_path}'")

# 2. Export Structured Summary to JSON
json_path = output_dir / 'simulation_summary.json'
summary = {
    'design_name': 'Inset-Fed Patch Antenna',
    'num_frequency_points': int(len(freqs_ghz)),
    'frequency_range_ghz': [float(np.min(freqs_ghz)), float(np.max(freqs_ghz))],
    'resonance_frequency_ghz': float(freqs_ghz[np.argmin(s11_db)]) if len(s11_db) > 0 else None,
    'minimum_s11_db': float(np.min(s11_db)) if len(s11_db) > 0 else None,
    'minimum_vswr': float(np.min(vswr)) if len(vswr) > 0 else None,
    'impedance_at_resonance': {
        'R_Ohm': float(r_in[np.argmin(s11_db)]) if len(s11_db) > 0 else None,
        'X_Ohm': float(x_in[np.argmin(s11_db)]) if len(s11_db) > 0 else None,
    },
    'units': {'frequency': 'GHz', 'return_loss': 'dB', 'impedance': 'Ohm'},
}
json_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f"Exported Simulation Summary JSON: '{json_path}'")

# 3. Export Standard Touchstone 1-Port (.s1p) Format
s1p_path = output_dir / 'antenna_s11.s1p'
with open(s1p_path, 'w', encoding='utf-8') as f:
    f.write('! Standard Touchstone 1-Port File generated by ANTON\n')
    f.write('! Format: Frequency (GHz) | S11 Real | S11 Imaginary | Reference Impedance: 50 Ohm\n')
    f.write('# GHz S RI R 50.0\n')
    for i in range(len(freqs_ghz)):
        re_val = float(np.real(s11_complex[i])) if len(s11_complex) > i else 0.0
        im_val = float(np.imag(s11_complex[i])) if len(s11_complex) > i else 0.0
        f.write(f'{freqs_ghz[i]:.6f} {re_val:.8f} {im_val:.8f}\n')
print(f"Exported Standard Touchstone S1P: '{s1p_path}'")
print('All simulation results exported successfully.')

## 13. Project Save & Clean Session Release
Saves the ANSYS HFSS project file and cleanly releases the PyAEDT desktop session handle.

In [ ]:
# Save project and release desktop handle
if app is not None:
    try:
        app.save_project()
        print(f"HFSS Project '{app.project_name}' saved successfully.")
        # Release desktop handle without forcibly closing the user's GUI session
        app.release_desktop(close_projects=False, close_desktop=False)
        print("Released PyAEDT desktop handle cleanly.")
    except Exception as e:
        print(f"Project save note: {e}")
else:
    print("Offline mode: No active HFSS session to save.")
